# Fashion MNIST Classification with Keras
## AI Expert Developer Course - L36

In this notebook we will:
1. Load the Fashion MNIST dataset
2. Preprocess the data (normalize, flatten, one-hot encode)
3. Visualize samples from the dataset
4. Build a Fully Connected Neural Network
5. Train the model and check for overfitting
6. Test with unseen images
7. Show beautiful visualizations and a Confusion Matrix

---
## Part 1: Import Libraries and Load the Data

**What this does:**
- We import all the libraries we need: TensorFlow/Keras for building the neural network, NumPy for math, Matplotlib and Seaborn for plots.
- We load the Fashion MNIST dataset directly from Keras. It comes already split into training (60,000 images) and test (10,000 images).
- Each image is 28x28 pixels in grayscale, showing one of 10 clothing categories.

In [ ]:
# Import all required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set a nice style for all our plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

In [ ]:
# Load Fashion MNIST dataset from Keras
# This gives us training and test sets already split
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Define the 10 clothing categories
class_names = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Let's see the shape of our data
print(f"Training set shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")
print(f"\nPixel value range: {X_train.min()} to {X_train.max()}")
print(f"Number of classes: {len(class_names)}")
print(f"Classes: {class_names}")

---
## Part 2: Preprocessing the Data

**What this does:**

We do 3 important preprocessing steps:

1. **Normalization** - Pixel values go from 0-255 to 0-1. This helps the neural network learn faster because smaller numbers are easier to work with.

2. **Flattening** - Each 28x28 image (a 2D grid) becomes a single row of 784 numbers. A Fully Connected network needs a flat input, not a grid.

3. **One-Hot Encoding** - Instead of a label like `3` (meaning 'Dress'), we create a vector like `[0,0,0,1,0,0,0,0,0,0]`. This helps the network understand that categories are separate - not that 'Dress' (3) is somehow bigger than 'T-shirt' (0).

In [ ]:
# --- STEP 1: Normalization ---
# Divide by 255 to scale pixel values from [0, 255] to [0, 1]
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

print("After Normalization:")
print(f"  Pixel value range: {X_train_norm.min():.1f} to {X_train_norm.max():.1f}")

# --- STEP 2: Flattening ---
# Reshape from (60000, 28, 28) to (60000, 784)
X_train_flat = X_train_norm.reshape(-1, 28 * 28)
X_test_flat = X_test_norm.reshape(-1, 28 * 28)

print(f"\nAfter Flattening:")
print(f"  Training shape: {X_train_flat.shape}  (each image is now a row of 784 numbers)")
print(f"  Test shape: {X_test_flat.shape}")

# --- STEP 3: One-Hot Encoding ---
# Convert label 3 -> [0,0,0,1,0,0,0,0,0,0]
y_train_onehot = to_categorical(y_train, num_classes=10)
y_test_onehot = to_categorical(y_test, num_classes=10)

print(f"\nAfter One-Hot Encoding:")
print(f"  Original label: {y_train[0]} ({class_names[y_train[0]]})")
print(f"  One-hot vector: {y_train_onehot[0]}")
print(f"  Labels shape: {y_train_onehot.shape}")

---
## Part 3: Visualize the Data (4x4 Grid)

**What this does:**
- We show a 4x4 grid of 16 random images from the training set.
- Each image has its class name as a title.
- This helps us understand what the data looks like before training.
- We also show the distribution of classes to check if the dataset is balanced.

In [ ]:
# Show a 4x4 grid of random training images
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle('Fashion MNIST - 16 Random Training Samples', fontsize=16, fontweight='bold')

# Pick 16 random indices
random_indices = np.random.choice(len(X_train), 16, replace=False)

for i, ax in enumerate(axes.flat):
    idx = random_indices[i]
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(f'{class_names[y_train[idx]]}', fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Show the distribution of classes in training data
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Training set distribution
unique, counts = np.unique(y_train, return_counts=True)
colors = sns.color_palette('husl', 10)
bars1 = ax1.bar([class_names[i] for i in unique], counts, color=colors)
ax1.set_title('Training Set - Class Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Class')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)
for bar, count in zip(bars1, counts):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
             f'{count}', ha='center', va='bottom', fontweight='bold')

# Test set distribution
unique_t, counts_t = np.unique(y_test, return_counts=True)
bars2 = ax2.bar([class_names[i] for i in unique_t], counts_t, color=colors)
ax2.set_title('Test Set - Class Distribution', fontsize=14, fontweight='bold')
ax2.set_xlabel('Class')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=45)
for bar, count in zip(bars2, counts_t):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
             f'{count}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("The dataset is perfectly balanced - each class has exactly 6,000 training and 1,000 test samples.")

---
## Part 4: Build the Fully Connected Neural Network

**What this does:**

We build a Fully Connected (Dense) neural network with this architecture:

| Layer | Neurons | Activation | Purpose |
|-------|---------|------------|----------|
| Input | 784 | - | Takes the flattened 28x28 image |
| Hidden 1 | 512 | ReLU | Learns basic features (edges, shapes) |
| Dropout 1 | - | - | Drops 30% of neurons to prevent overfitting |
| Hidden 2 | 256 | ReLU | Learns more complex patterns |
| Dropout 2 | - | - | Drops 30% of neurons to prevent overfitting |
| Hidden 3 | 128 | ReLU | Learns high-level features |
| Dropout 3 | - | - | Drops 20% of neurons to prevent overfitting |
| Output | 10 | Softmax | Gives probability for each of the 10 classes |

**Key choices explained:**

- **Loss function: Categorical Crossentropy** - This is the standard loss for multi-class classification. It measures how far our predicted probabilities are from the true labels. Lower = better.

- **Optimizer: Adam (learning rate = 0.001)** - Adam is an adaptive optimizer that adjusts the learning step automatically. The learning rate of 0.001 is a good default - not too fast (might miss the best answer) and not too slow.

- **Learning rate** - This controls how big each step is when the network adjusts its weights. Think of it like walking toward a target: too big and you overshoot, too small and it takes forever.

- **Activation: ReLU** - Rectified Linear Unit. It outputs 0 for negative values and the value itself for positive values. Simple and fast.

- **Activation: Softmax (output)** - Converts the output into probabilities that sum to 1. So the network tells us "80% chance this is a Sneaker, 15% Ankle boot, 5% Sandal".

- **Dropout** - Randomly turns off some neurons during training. This forces the network to not rely on any single neuron, making it more robust.

In [ ]:
# Build the model
model = keras.Sequential([
    # Input layer - takes 784 features (flattened 28x28 image)
    layers.Input(shape=(784,)),

    # Hidden Layer 1: 512 neurons with ReLU activation
    layers.Dense(512, activation='relu', name='hidden_1'),
    layers.Dropout(0.3, name='dropout_1'),   # Drop 30% to prevent overfitting

    # Hidden Layer 2: 256 neurons with ReLU activation
    layers.Dense(256, activation='relu', name='hidden_2'),
    layers.Dropout(0.3, name='dropout_2'),   # Drop 30%

    # Hidden Layer 3: 128 neurons with ReLU activation
    layers.Dense(128, activation='relu', name='hidden_3'),
    layers.Dropout(0.2, name='dropout_3'),   # Drop 20%

    # Output Layer: 10 neurons (one per class) with Softmax
    layers.Dense(10, activation='softmax', name='output')
])

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),  # Adam optimizer, step size 0.001
    loss='categorical_crossentropy',                        # Loss for multi-class classification
    metrics=['accuracy']                                    # Track accuracy during training
)

# Show the model summary
print("=" * 65)
print("MODEL ARCHITECTURE SUMMARY")
print("=" * 65)
model.summary()

print(f"\n{'=' * 65}")
print(f"Total layers: 3 hidden + 1 output = 4 Dense layers")
print(f"Total neurons: 512 + 256 + 128 + 10 = 906")
print(f"Dropout layers: 3 (to prevent overfitting)")
print(f"Loss function: Categorical Crossentropy")
print(f"Optimizer: Adam (learning rate = 0.001)")
print(f"Output activation: Softmax (probabilities for 10 classes)")
print(f"{'=' * 65}")

---
## Part 5: Train the Model (Fit) and Check for Overfitting

**What this does:**
- We train the model for 30 epochs (30 passes through the entire training data).
- We use a batch size of 128 (the network sees 128 images at a time before updating weights).
- We use 15% of training data as validation to monitor overfitting.
- **Overfitting check**: If the training loss keeps going down but validation loss starts going UP, that means overfitting. We use EarlyStopping to stop training if validation loss doesn't improve for 5 epochs.
- We plot training vs validation loss and accuracy to visually check for overfitting.

In [ ]:
# Set up callbacks
# EarlyStopping: stop training if validation loss doesn't improve for 5 epochs
# ReduceLROnPlateau: reduce learning rate if validation loss plateaus
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

# Train the model
print("Starting training...\n")
history = model.fit(
    X_train_flat, y_train_onehot,
    epochs=30,
    batch_size=128,
    validation_split=0.15,         # Use 15% of training data for validation
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete!")

In [ ]:
# Plot training history - Loss and Accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- Loss Plot ---
ax1.plot(history.history['loss'], label='Training Loss', linewidth=2, color='#2196F3')
ax1.plot(history.history['val_loss'], label='Validation Loss', linewidth=2, color='#FF5722', linestyle='--')
ax1.set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss (Categorical Crossentropy)', fontsize=12)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Mark the best epoch
best_epoch = np.argmin(history.history['val_loss'])
best_val_loss = history.history['val_loss'][best_epoch]
ax1.axvline(x=best_epoch, color='green', linestyle=':', alpha=0.7)
ax1.annotate(f'Best: epoch {best_epoch+1}\nLoss: {best_val_loss:.4f}',
             xy=(best_epoch, best_val_loss), fontsize=10,
             xytext=(best_epoch + 2, best_val_loss + 0.05),
             arrowprops=dict(arrowstyle='->', color='green'),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.7))

# --- Accuracy Plot ---
ax2.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2, color='#2196F3')
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2, color='#FF5722', linestyle='--')
ax2.set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

# Mark best accuracy
best_acc_epoch = np.argmax(history.history['val_accuracy'])
best_val_acc = history.history['val_accuracy'][best_acc_epoch]
ax2.axvline(x=best_acc_epoch, color='green', linestyle=':', alpha=0.7)
ax2.annotate(f'Best: epoch {best_acc_epoch+1}\nAcc: {best_val_acc:.4f}',
             xy=(best_acc_epoch, best_val_acc), fontsize=10,
             xytext=(best_acc_epoch + 2, best_val_acc - 0.03),
             arrowprops=dict(arrowstyle='->', color='green'),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.7))

plt.tight_layout()
plt.show()

# Check for overfitting
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]
gap = final_val_loss - final_train_loss

print(f"\n{'=' * 50}")
print(f"OVERFITTING CHECK")
print(f"{'=' * 50}")
print(f"Final Training Loss:   {final_train_loss:.4f}")
print(f"Final Validation Loss: {final_val_loss:.4f}")
print(f"Gap: {gap:.4f}")
if gap < 0.1:
    print(f"Result: No significant overfitting! The gap is small.")
else:
    print(f"Result: Some overfitting detected. Consider more dropout or regularization.")
print(f"{'=' * 50}")

---
## Part 6: Test with Unseen Images

**What this does:**
- First, we evaluate the model on the entire test set (10,000 images the model has never seen).
- Then we pick random individual test images and show what the model predicts vs the true label.
- For each image, we also show the confidence (probability) for all 10 classes as a bar chart.

In [ ]:
# Evaluate on the full test set
test_loss, test_accuracy = model.evaluate(X_test_flat, y_test_onehot, verbose=0)

print(f"{'=' * 50}")
print(f"TEST SET RESULTS")
print(f"{'=' * 50}")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"{'=' * 50}")

In [ ]:
# Show predictions for 8 random unseen test images
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Model Predictions on Unseen Test Images', fontsize=18, fontweight='bold')

random_test_indices = np.random.choice(len(X_test), 8, replace=False)

for i, ax in enumerate(axes.flat):
    idx = random_test_indices[i]
    image = X_test[idx]
    true_label = y_test[idx]

    # Get prediction
    prediction = model.predict(X_test_flat[idx:idx+1], verbose=0)
    predicted_label = np.argmax(prediction)
    confidence = np.max(prediction) * 100

    # Show image
    ax.imshow(image, cmap='gray')

    # Color title green if correct, red if wrong
    is_correct = predicted_label == true_label
    color = 'green' if is_correct else 'red'
    symbol = 'V' if is_correct else 'X'

    ax.set_title(
        f'True: {class_names[true_label]}\n'
        f'Pred: {class_names[predicted_label]} ({confidence:.1f}%) {symbol}',
        fontsize=11, fontweight='bold', color=color
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Detailed prediction with confidence bars for 4 images
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
fig.suptitle('Detailed Predictions with Confidence Scores', fontsize=18, fontweight='bold')

random_indices_detail = np.random.choice(len(X_test), 4, replace=False)

for i in range(4):
    idx = random_indices_detail[i]
    image = X_test[idx]
    true_label = y_test[idx]

    # Get prediction probabilities
    prediction = model.predict(X_test_flat[idx:idx+1], verbose=0)[0]
    predicted_label = np.argmax(prediction)

    # Left: Show the image
    axes[i, 0].imshow(image, cmap='gray')
    is_correct = predicted_label == true_label
    color = 'green' if is_correct else 'red'
    axes[i, 0].set_title(
        f'True: {class_names[true_label]} | Pred: {class_names[predicted_label]}',
        fontsize=12, fontweight='bold', color=color
    )
    axes[i, 0].axis('off')

    # Right: Show confidence bars
    bar_colors = ['green' if j == true_label else 'red' if j == predicted_label and not is_correct else 'steelblue'
                  for j in range(10)]
    bars = axes[i, 1].barh(class_names, prediction * 100, color=bar_colors)
    axes[i, 1].set_xlim(0, 100)
    axes[i, 1].set_xlabel('Confidence (%)', fontsize=10)
    axes[i, 1].set_title('Class Probabilities', fontsize=12, fontweight='bold')

    # Add percentage labels on bars
    for bar, prob in zip(bars, prediction):
        if prob > 0.01:
            axes[i, 1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                           f'{prob*100:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## Part 7: Full Visualization - Confusion Matrix and Per-Class Results

**What this does:**
- **Confusion Matrix**: A grid that shows, for each true class, how many times the model predicted each class. The diagonal shows correct predictions. Off-diagonal shows mistakes.
- **Per-class accuracy**: Bar chart showing how accurate the model is for each clothing type.
- **Classification report**: Precision, Recall, and F1-score for each class.
- **Misclassified examples**: We find and show images that the model got WRONG, so we can see where it struggles.

In [ ]:
# Get predictions for the entire test set
y_pred_probs = model.predict(X_test_flat, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Create Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# Plot a beautiful confusion matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 9))

# Raw counts confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=ax1, cbar_kws={'label': 'Count'})
ax1.set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Predicted Label', fontsize=12)
ax1.set_ylabel('True Label', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
ax1.tick_params(axis='y', rotation=0)

# Normalized confusion matrix (percentages)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_normalized, annot=True, fmt='.1f', cmap='RdYlGn',
            xticklabels=class_names, yticklabels=class_names,
            ax=ax2, cbar_kws={'label': 'Percentage (%)'})
ax2.set_title('Confusion Matrix (Normalized %)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Predicted Label', fontsize=12)
ax2.set_ylabel('True Label', fontsize=12)
ax2.tick_params(axis='x', rotation=45)
ax2.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy bar chart
per_class_accuracy = cm.diagonal() / cm.sum(axis=1) * 100

fig, ax = plt.subplots(figsize=(14, 6))

colors = ['#4CAF50' if acc >= 90 else '#FF9800' if acc >= 80 else '#F44336'
          for acc in per_class_accuracy]

bars = ax.bar(class_names, per_class_accuracy, color=colors, edgecolor='black', linewidth=0.5)

# Add value labels on bars
for bar, acc in zip(bars, per_class_accuracy):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Per-Class Accuracy on Test Set', fontsize=16, fontweight='bold')
ax.set_xlabel('Class', fontsize=13)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_ylim(0, 105)
ax.tick_params(axis='x', rotation=45)
ax.axhline(y=test_accuracy * 100, color='blue', linestyle='--', alpha=0.7,
           label=f'Overall Accuracy: {test_accuracy*100:.1f}%')
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

# Color legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4CAF50', label='90%+ (Great)'),
    Patch(facecolor='#FF9800', label='80-90% (Good)'),
    Patch(facecolor='#F44336', label='<80% (Needs improvement)')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print("=" * 70)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 70)
print(classification_report(y_test, y_pred, target_names=class_names))
print("=" * 70)
print(f"\nOverall Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Total Test Samples: {len(y_test)}")
print(f"Correct Predictions: {np.sum(y_pred == y_test)}")
print(f"Wrong Predictions: {np.sum(y_pred != y_test)}")

In [ ]:
# Show misclassified examples - where the model makes mistakes
misclassified_indices = np.where(y_pred != y_test)[0]
print(f"Total misclassified images: {len(misclassified_indices)} out of {len(y_test)}")

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Misclassified Images - Where the Model Struggles',
             fontsize=16, fontweight='bold', color='red')

random_wrong = np.random.choice(misclassified_indices, min(12, len(misclassified_indices)), replace=False)

for i, ax in enumerate(axes.flat):
    if i < len(random_wrong):
        idx = random_wrong[i]
        ax.imshow(X_test[idx], cmap='gray')
        confidence = np.max(y_pred_probs[idx]) * 100
        ax.set_title(
            f'True: {class_names[y_test[idx]]}\n'
            f'Pred: {class_names[y_pred[idx]]} ({confidence:.0f}%)',
            fontsize=10, fontweight='bold', color='red'
        )
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Final Summary Dashboard
fig = plt.figure(figsize=(20, 8))
fig.suptitle('Fashion MNIST Classification - Final Results Dashboard',
             fontsize=20, fontweight='bold', y=1.02)

# Plot 1: Training vs Validation accuracy
ax1 = fig.add_subplot(1, 3, 1)
epochs_range = range(1, len(history.history['accuracy']) + 1)
ax1.fill_between(epochs_range, history.history['accuracy'], alpha=0.3, color='#2196F3')
ax1.fill_between(epochs_range, history.history['val_accuracy'], alpha=0.3, color='#FF5722')
ax1.plot(epochs_range, history.history['accuracy'], label='Train', linewidth=2, color='#2196F3')
ax1.plot(epochs_range, history.history['val_accuracy'], label='Val', linewidth=2, color='#FF5722')
ax1.set_title('Accuracy Over Time', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Per-class accuracy radar/bar
ax2 = fig.add_subplot(1, 3, 2)
sorted_idx = np.argsort(per_class_accuracy)
colors_sorted = ['#4CAF50' if per_class_accuracy[i] >= 90
                 else '#FF9800' if per_class_accuracy[i] >= 80
                 else '#F44336' for i in sorted_idx]
ax2.barh([class_names[i] for i in sorted_idx], per_class_accuracy[sorted_idx],
         color=colors_sorted, edgecolor='black', linewidth=0.5)
ax2.set_xlim(70, 100)
ax2.set_title('Per-Class Accuracy (Sorted)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Accuracy (%)')
for i, (acc, name_idx) in enumerate(zip(per_class_accuracy[sorted_idx], sorted_idx)):
    ax2.text(acc + 0.3, i, f'{acc:.1f}%', va='center', fontweight='bold', fontsize=9)

# Plot 3: Correct vs Wrong pie chart
ax3 = fig.add_subplot(1, 3, 3)
correct = np.sum(y_pred == y_test)
wrong = np.sum(y_pred != y_test)
ax3.pie([correct, wrong], labels=[f'Correct\n{correct}', f'Wrong\n{wrong}'],
        autopct='%1.1f%%', colors=['#4CAF50', '#F44336'],
        explode=(0.05, 0.05), shadow=True, startangle=90,
        textprops={'fontsize': 12, 'fontweight': 'bold'})
ax3.set_title('Overall Prediction Results', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("PROJECT COMPLETE!")
print("=" * 60)
print(f"Final Test Accuracy: {test_accuracy*100:.2f}%")
print(f"The model successfully classifies fashion items into 10 categories.")
print("=" * 60)